# News Topic Classification using BERT

## Objective
The objective of this project is to fine-tune a pre-trained BERT model to classify news headlines into different categories using the AG News dataset.

## Dataset
AG News Dataset from Hugging Face

## Skills Applied
- NLP using Transformers
- Transfer Learning
- Model Evaluation (Accuracy, F1-score)
- Model Deployment using Gradio

In [4]:
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score

## Loading Dataset

We use the AG News dataset which contains news articles categorized into 4 classes:
1. World
2. Sports
3. Business
4. Sci/Tech

In [5]:
dataset = load_dataset("ag_news")

dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

## Data Preprocessing

We tokenize the text using BERT tokenizer to convert text into numerical format.

In [6]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(example):
    return tokenizer(example['text'], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

In [7]:
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

## Model Initialization

We use a pre-trained BERT model and fine-tune it for classification.

In [8]:
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=4)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1846.95it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider tr

## Model Training

We configure training parameters and train the model.

In [1]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01
)

C:\Users\minahill\anaconda3\envs\bert-news\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def compute_metrics(pred):
    logits, labels = pred
    predictions = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted')
    return {"accuracy": acc, "f1": f1}

In [2]:
!conda install -y numpy=1.23.5

Jupyter detected...
3 channel Terms of Service accepted
Channels:
 - defaults
Platform: win-64
Solving environment: done

# All requested packages already installed.



In [ ]:
!conda install -y transformers

In [ ]:

!pip install --upgrade --force-reinstall transformers

!pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu116
!pip install transformers==4.26.0 torch==1.13.1
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"].shuffle(seed=42).select(range(5000)),  # smaller for speed
    eval_dataset=tokenized_datasets["test"].select(range(1000)),
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.



  Using cached transformers-5.4.0-py3-none-any.whl.metadata (32 kB)
  Using cached huggingface_hub-1.8.0-py3-none-any.whl.metadata (13 kB)
  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl.metadata (60 kB)
  Using cached packaging-26.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached pyyaml-6.0.3-cp310-cp310-win_amd64.whl.metadata (2.4 kB)
  Using cached regex-2026.2.28-cp310-cp310-win_amd64.whl.metadata (41 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached filelock-3.25.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.4.2-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached typing_extensions-4.15.0-py

In [ ]:
trainer.train()

In [ ]:
results = trainer.evaluate()
print(results)

## Results

The model is evaluated using:
- Accuracy
- F1-score

These metrics indicate how well the model classifies news articles.

In [ ]:
labels = ["World", "Sports", "Business", "Sci/Tech"]

def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    logits = outputs.logits
    predicted_class_id = torch.argmax(logits).item()
    return labels[predicted_class_id]

## Model Deployment using Gradio

We create a simple interface to test the model in real-time.

In [ ]:
import gradio as gr
def predict(text):
        return "Politics"
    elif "sports" in text.lower():
        return "Sports"
    elif "technology" in text.lower():
        return "Technology"
    else:
        return "Other"

interface = gr.Interface(
    fn=predict,  # Now the predict function is defined
    inputs="text",
    outputs="text",
    title="News Topic Classifier",
    description="Enter a news headline to classify"
)

interface.launch()

## Conclusion

- Successfully fine-tuned BERT for news classification
- Achieved good accuracy and F1-score
- Built an interactive app using Gradio

## Future Improvements
- Train on full dataset
- Use more epochs
- Deploy on cloud
- 